# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: /workspaces/data-analysis-course/.venv/bin/python
pandas: 2.2.2
Рабочая папка: /workspaces/data-analysis-course/lessons/01-introduction/work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
home_denied = home_outcomes.eq(3).sum()
home_denial_rate = home_denied / home_n

print("Условие: action_taken == 3")
print("Числитель:", home_denied)
print("Знаменатель:", home_n)
print("Доля:", home_denial_rate)
print(f"Процент: {home_denial_rate * 100:.2f}%")

home_approved = home_outcomes.isin([1, 2]).sum()
assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1

Условие: action_taken == 3
Числитель: 691
Знаменатель: 6830
Доля: 0.10117130307467057
Процент: 10.12%


## H2. Одобрение без принятия предложения

In [4]:
home_not_accepted = home_outcomes.eq(2).sum()
home_not_accepted_rate = home_not_accepted / home_n
print("Условие: action_taken == 2")
print("Числитель:", home_not_accepted, "| Знаменатель:", home_n)
print(f"Процент: {home_not_accepted_rate * 100:.2f}%")

home_issued = home_outcomes.eq(1).sum()
assert home_approved - home_issued == home_not_accepted   # по счётчикам

approved_rate = home_approved / home_n
issued_rate = home_issued / home_n
print("Одобрения:", home_approved, f"{approved_rate * 100:.2f}%")
print("Выдачи:", home_issued, f"{issued_rate * 100:.2f}%")
print(f"Разность: {(approved_rate - issued_rate) * 100:.2f} п.п.")

Условие: action_taken == 2
Числитель: 183 | Знаменатель: 6830
Процент: 2.68%
Одобрения: 6139 89.88%
Выдачи: 5956 87.20%
Разность: 2.68 п.п.


## H3. Объяснение инструментов

В отчёте

## Дополнительная часть

## Исходный снимок для H+1

In [5]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

In [6]:
raw_issued = raw_outcomes.eq(1).sum()
raw_issued_rate = raw_issued / raw_n
home_issued_rate = home_issued / home_n   # home_issued из H2

print("raw:", raw_issued, "/", raw_n, f"= {raw_issued_rate * 100:.2f}%")
print("decisions:", home_issued, "/", home_n, f"= {home_issued_rate * 100:.2f}%")
assert raw_issued == home_issued

raw: 5956 / 8575 = 69.46%
decisions: 5956 / 6830 = 87.20%


## Текстовые коды для H+2

In [7]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом

In [8]:
as_int = text_example["action_taken"].eq(1)
as_text = text_example["action_taken"].eq("1")
print(as_int.tolist(), as_int.sum())
print(as_text.tolist(), as_text.sum())

[False, False, False, False, False, False] 0
[True, True, False, False, False, False] 2
